# Cash Liquidity Forecasting — Transaction-Level Time Series EDA
**Data:** ~1.5 years of AR line items (SAP-style) pulled from SQL into Databricks/pandas
**Goal:** understand the data well enough to build a cash forecasting model, and collect everything needed for a final report.

### Table of contents
| # | Section | Question it answers |
|---|---------|--------------------|
| 0 | Setup & config | Where do I change column names / paths? |
| 1 | Load data | Is the data in memory correctly? |
| 2 | Profiling | What does the raw table look like? Constants, nulls, dtypes |
| 3 | Cleaning | Types, sentinel dates, duplicates, standardised codes |
| 4 | Integrity checks | Do dates / amounts / clearing info make logical sense? |
| 5 | Feature engineering | Payment terms, delay, calendar, ageing features |
| 6 | Categorical profile | Business_Area, Posting_Key, GL_Document_Type, HBKID |
| 7 | Amount analysis | Distribution, sign, outliers, concentration |
| 8 | Time-series construction | Posting vs Due vs Clearing series; completeness; censoring |
| 9 | Trend & seasonality | Weekday, month-end, STL, monthly pattern |
| 10 | Statistical tests | Stationarity, ACF/PACF, strengths, intermittency |
| 11 | Payment behaviour | Delay distribution, collection curve, customer behaviour |
| 12 | Customer analysis | Pareto, concentration, activity, demand class |
| 13 | Segment series | Business area / doc type / HBKID level series |
| 14 | Lead-lag & AR balance | Due → Clearing relationship, open-item balance |
| 15 | Anomalies | Spikes, large items, suspicious duplicates |
| 16 | Baselines | Benchmarks any model must beat |
| 17 | Modelling readiness | Auto-generated findings and recommendations |
| 18 | Report export | HTML + Excel + JSON from the collected variables |

### Key mental model (read once)
In AR cash forecasting there are **three different clocks** on every invoice line:
1. **Posting_Date** → when the invoice was raised (billing activity).
2. **Due_Date** → when cash is *contractually expected*.
3. **Clearing_Posting_Date** → when cash *actually arrived* (this is what we forecast).

`payment_delay = Clearing − Due` is the heart of the problem. Forecasting cash = forecasting **how much falls due** × **how late it gets paid**.

> **Databricks tip:** run `%pip install statsmodels seaborn openpyxl` in a first cell if any import fails, then `dbutils.library.restartPython()`.

## 0. Setup & configuration

In [ ]:
import warnings, json, base64, io, datetime as dt
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller, kpss, acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100, "display.width", 200, "display.float_format", lambda x: f"{x:,.2f}")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.figsize": (12, 4), "axes.titlesize": 12, "axes.titleweight": "bold"})

In [ ]:
# ───────── CONFIG: change only this cell if your column names / paths differ ─────────
CFG = dict(
    USE_SAMPLE_DATA = False,                      # True = synthetic data to dry-run the notebook
    SQL_QUERY       = "SELECT * FROM your_catalog.your_schema.your_table",   # <-- EDIT
    OUTPUT_DIR      = "eda_outputs",              # on Databricks consider "/dbfs/FileStore/eda_outputs"

    # core columns
    POSTING   = "Posting_Date",
    DUE       = "Due_Date",
    CLEARING  = "Clearing_Posting_Date",
    BASELINE  = "Baseline_Date_for_Due_Date_Calc",
    AMOUNT    = "Amount_in_LC",                   # amount we forecast (local currency)
    CUSTOMER  = "Customer_Number",
    CLEAR_DOC = "Clearing_Document",
    DOC_NO    = "Accounting_Document_Number",

    CAT_COLS  = ["Business_Area", "Posting_Key", "GL_Document_Type", "HBKID"],
    PAD       = {"Business_Area": 4, "Posting_Key": 2},     # zero-padding of coded columns
    TOP_N     = 10,
    HOLDOUT_WEEKS = 8,                            # last N full weeks used for baseline evaluation
    MATURE_DAYS   = 90,                           # items due >= N days before data end are "mature" (censoring-safe)
    SENTINEL_BEFORE = "1990-01-01",               # dates earlier than this are treated as missing
)
P, D, C, B = CFG["POSTING"], CFG["DUE"], CFG["CLEARING"], CFG["BASELINE"]
AMT, CUST  = CFG["AMOUNT"], CFG["CUSTOMER"]
CAT        = CFG["CAT_COLS"]
OUT = Path(CFG["OUTPUT_DIR"]); FIG_DIR = OUT / "figures"; FIG_DIR.mkdir(parents=True, exist_ok=True)

# ───────── REPORT STORE: everything that goes into the final report is registered here ─────────
REPORT = {"meta": {"created": str(dt.datetime.now()), "config": {k: v for k, v in CFG.items() if k != "SQL_QUERY"}},
          "kpis": {}, "insights": [], "tables": {}, "figures": []}

def save_fig(name, caption="", section=""):
    """Save + register the current matplotlib figure for the report, then show it."""
    path = FIG_DIR / f"{name}.png"
    plt.tight_layout(); plt.savefig(path, dpi=130, bbox_inches="tight")
    REPORT["figures"].append({"name": name, "path": str(path), "caption": caption, "section": section})
    plt.show()

def add_table(name, frame, note="", section="", index=True):
    REPORT["tables"][name] = {"df": frame.copy(), "note": note, "section": section, "index": index}
    return frame

def add_insight(section, text, level="info"):
    """level: info | warning | action"""
    REPORT["insights"].append({"section": section, "level": level, "text": text})
    icon = {"info": "ℹ️", "warning": "⚠️", "action": "➡️"}[level]
    print(f"{icon} [{section}] {text}")

def kpi(name, value):
    REPORT["kpis"][name] = value.item() if hasattr(value, "item") else value

## 1. Load data
Pull from SQL via Spark and convert to pandas. For ~1.5 years of line items this is typically small enough for `toPandas()`.
Set `USE_SAMPLE_DATA=True` in the config to dry-run the whole notebook on synthetic data first.

In [ ]:
def make_sample_data(n=6000, seed=42):
    rng = np.random.default_rng(seed)
    data_end = pd.Timestamp("2026-06-30")
    post = pd.Timestamp("2025-01-01") + pd.to_timedelta(rng.integers(0, 545, n), unit="D")
    custs = np.arange(100100, 100139)
    cust = rng.choice(custs, n, p=rng.dirichlet(np.ones(len(custs)) * 0.35))
    bias = {c: rng.normal(6, 9) for c in custs}
    terms = rng.choice([30, 45, 60], n, p=[.5, .3, .2])
    base = post + pd.to_timedelta(rng.choice([0, 0, 0, 3, 7], n), unit="D")
    due = base + pd.to_timedelta(terms, unit="D")
    delay = np.array([bias[c] for c in cust]) + rng.gamma(2, 3, n) - 5
    clear = due + pd.to_timedelta(np.round(delay), unit="D")
    clear = pd.Series(clear).clip(lower=pd.Series(post) + pd.Timedelta(days=1))
    # payments mostly on business days
    clear = clear.where(clear.dt.dayofweek < 5, clear + pd.to_timedelta(2, unit="D"))
    clear = clear.where(clear <= data_end, pd.NaT)
    doc_type = rng.choice(["F2", "AB", "DR", "ZM", "S2", "ZP"], n, p=[.6, .1, .1, .08, .07, .05])
    amt = np.round(rng.lognormal(9, 1.1, n), 2)
    amt = np.where(np.isin(doc_type, ["DR", "ZM"]), -amt * 0.2, amt)
    cleared = clear.notna().values
    return pd.DataFrame({
        "GL_Account": "1200000", "Company_Code_ERP": "1000", "DR_CR_Indicator": "S", "Account_Type": "D",
        "Company_ERP_Currency_Code": "INR",
        "Business_Area": rng.choice(["0131", "0019"], n, p=[.7, .3]),
        "Posting_Key": rng.choice(["01", "05", "07"], n, p=[.8, .1, .1]),
        "GL_Document_Type": doc_type, "Posting_Month": post.month,
        "Customer_Number": cust.astype(str), "Due_date_week": due.isocalendar().week.values,
        "HBKID": rng.choice([f"HB{i:03d}" for i in range(73)], n),
        "Clearing_Posting_Date": clear.values, "GL_Item_Text": rng.choice([f"TXT{i}" for i in range(492)], n),
        "Baseline_Date_for_Due_Date_Calc": base, "Due_Date": due, "Posting_Date": post,
        "Clearing_Document": np.where(cleared, rng.integers(1_000_000, 1_001_500, n).astype(str), None),
        "Amount_in_LC": amt, "Amount_in_DC": amt, "Amount_in_LC_DR_CR": amt,
        "Account_ID": rng.integers(1, 6014, n).astype(str),
        "Accounting_Document_Reference": rng.integers(1, 6182, n).astype(str),
        "Accounting_Document_Number": rng.integers(1, 6261, n).astype(str),
    })

if CFG["USE_SAMPLE_DATA"]:
    df_raw = make_sample_data()
else:
    sdf = spark.sql(CFG["SQL_QUERY"])          # `spark` is provided by Databricks
    df_raw = sdf.toPandas()

print(f"Loaded {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
df_raw.head()

## 2. Raw data profiling
**Why:** before any modelling we need to know the shape, types, missingness and which columns carry information.
Columns with **1 unique value** (e.g. `GL_Account`, `Company_Code_ERP`, `DR_CR_Indicator`, `Account_Type`, currency) are constants — they describe the *scope* of the dataset but are useless as features.

In [ ]:
profile = pd.DataFrame({
    "dtype": df_raw.dtypes.astype(str),
    "nunique": df_raw.nunique(dropna=True),
    "null_cnt": df_raw.isna().sum(),
    "null_pct": (df_raw.isna().mean() * 100).round(2),
}).sort_values("nunique")
profile["role_guess"] = np.select(
    [profile["nunique"] <= 1, profile["nunique"] <= 12, profile["nunique"] <= 100],
    ["constant (scope)", "low-cardinality category", "mid-cardinality category"], default="high-cardinality / id / numeric")
add_table("raw_profile", profile, "Column profile of the raw extract", "2. Profiling")

const_cols = profile.index[profile["nunique"] <= 1].tolist()
const_values = {c: df_raw[c].dropna().unique()[:1].tolist() for c in const_cols}
kpi("n_rows_raw", len(df_raw)); kpi("n_cols_raw", df_raw.shape[1]); kpi("constant_columns", const_values)
add_insight("2. Profiling", f"{len(const_cols)} constant columns (dataset scope): {const_values}. Drop from modelling features.")
profile

In [ ]:
# Nulls heatmap-style view for columns that have any nulls
nulls = profile[profile.null_cnt > 0]
if len(nulls):
    nulls["null_pct"].sort_values().plot.barh(figsize=(8, max(2, 0.3 * len(nulls))), color="tomato")
    plt.title("Null % by column"); plt.xlabel("%")
    save_fig("nulls_by_column", "Null percentage per column", "2. Profiling")
else:
    print("No nulls in any column.")

## 3. Data cleaning
Steps: strip whitespace → blank strings to null → parse dates (invalid / SAP sentinel dates such as `9999-12-31` → NaT) → zero-pad coded columns → numeric amounts → remove exact duplicates.
Every step is logged in `clean_log` so the report can show what was changed.

In [ ]:
df = df_raw.copy()
df.columns = [c.strip() for c in df.columns]
clean_log = []

# 1. strings: strip + blanks -> NA
for c in df.columns:
    if df[c].dtype == "object" or str(df[c].dtype).startswith("string"):
        s = df[c].astype("string").str.strip()
        n_blank = int(s.isin(["", "nan", "None", "NULL", "null"]).sum())
        df[c] = s.mask(s.isin(["", "nan", "None", "NULL", "null"]))
        if n_blank: clean_log.append((c, "blank/placeholder strings -> null", n_blank))

# 2. dates
for c in [P, D, C, B]:
    before = int(df[c].notna().sum())
    s = pd.to_datetime(df[c], errors="coerce")
    s = s.mask(s < pd.Timestamp(CFG["SENTINEL_BEFORE"]))
    s = s.dt.normalize()
    lost = before - int(s.notna().sum())
    df[c] = s
    clean_log.append((c, "parsed to datetime (invalid/sentinel -> NaT)", lost))

# 3. coded columns -> zero-padded strings
for c, w in CFG["PAD"].items():
    if c in df:
        df[c] = df[c].astype("string").str.replace(r"\.0$", "", regex=True).str.zfill(w)
for c in [CUST, "HBKID", CFG["CLEAR_DOC"], CFG["DOC_NO"]]:
    if c in df:
        df[c] = df[c].astype("string").str.replace(r"\.0$", "", regex=True)

# 4. numerics
for c in [x for x in ["Amount_in_LC", "Amount_in_DC", "Amount_in_LC_DR_CR"] if x in df]:
    before = int(df[c].notna().sum())
    df[c] = pd.to_numeric(df[c], errors="coerce")
    clean_log.append((c, "to numeric", before - int(df[c].notna().sum())))

# 5. exact duplicates
n_dup = int(df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
clean_log.append(("ALL", "exact duplicate rows removed", n_dup))

# 6. drop null key fields
for c in [P, D, AMT]:
    n_na = int(df[c].isna().sum())
    if n_na:
        clean_log.append((c, "rows dropped (null in key field)", n_na))
        df = df[df[c].notna()].reset_index(drop=True)

clean_log = pd.DataFrame(clean_log, columns=["column", "action", "rows_affected"])
add_table("cleaning_log", clean_log, "Cleaning actions performed", "3. Cleaning", index=False)
kpi("n_rows_clean", len(df)); kpi("duplicates_removed", n_dup)
clean_log

## 4. Integrity & logic checks
**Why:** time-series models trust the dates blindly. A clearing date before the posting date, or a due date before the baseline, silently corrupts delay features.
Each check below is a boolean mask; the summary table feeds the report.

In [ ]:
today = pd.Timestamp.today().normalize()
DATA_END = max(df[P].max(), df[C].max())       # last date on which we observe real activity
DATA_START = df[P].min()
kpi("data_start", str(DATA_START.date())); kpi("data_end", str(DATA_END.date()))
print(f"Observation window: {DATA_START.date()} -> {DATA_END.date()}  ({(DATA_END - DATA_START).days} days)")

checks = {
    "Due_Date < Posting_Date":                 df[D] < df[P],
    "Baseline > Due_Date":                     df[B] > df[D],
    "Baseline < Posting_Date":                 df[B] < df[P],
    "Clearing < Posting_Date":                 df[C] < df[P],
    "Clearing date missing (open items)":      df[C].isna(),
    "Clearing_Document present, no date":      df[CFG["CLEAR_DOC"]].notna() & df[C].isna(),
    "Clearing date present, no document":      df[CFG["CLEAR_DOC"]].isna() & df[C].notna(),
    "Posting_Date in future":                  df[P] > today,
    "Clearing_Date in future":                 df[C] > today,
    "Amount == 0":                             df[AMT] == 0,
    "Amount < 0 (credit / reversal)":          df[AMT] < 0,
    "Due on weekend":                          df[D].dt.dayofweek >= 5,
    "Clearing on weekend":                     df[C].dt.dayofweek >= 5,
}
if "Amount_in_LC_DR_CR" in df:
    checks["Sign(Amount_in_LC) != Sign(Amount_in_LC_DR_CR)"] = np.sign(df[AMT]) != np.sign(df["Amount_in_LC_DR_CR"])
if "Amount_in_DC" in df:
    checks["Sign(LC) != Sign(DC)"] = np.sign(df[AMT]) != np.sign(df["Amount_in_DC"])

integrity = pd.DataFrame({"rows": {k: int(v.sum()) for k, v in checks.items()}})
integrity["pct"] = (integrity.rows / len(df) * 100).round(2)
integrity["severity"] = np.where(integrity.index.str.contains("<|>|future|no date|no document|Sign", regex=True) & (integrity.rows > 0), "REVIEW", "info")
add_table("integrity_checks", integrity, "Logical consistency checks on dates, clearing info and amounts", "4. Integrity")
for k, r in integrity[integrity.severity == "REVIEW"].iterrows():
    add_insight("4. Integrity", f"'{k}': {r.rows:,} rows ({r.pct}%) — review before modelling.", "warning")
integrity

In [ ]:
# Date ranges per date column + amount currency consistency
ranges = pd.DataFrame({c: {"min": df[c].min(), "max": df[c].max(), "nulls": df[c].isna().sum()} for c in [P, D, C, B]}).T
add_table("date_ranges", ranges, "Min/max/nulls of the four date columns", "4. Integrity")
display(ranges)

# Implied FX rate LC/DC — should be stable if both currency amounts are populated
if "Amount_in_DC" in df:
    fx = (df[AMT] / df["Amount_in_DC"].replace(0, np.nan)).replace([np.inf, -np.inf], np.nan)
    print("Implied LC/DC rate:", fx.describe()[["min", "50%", "max"]].round(4).to_dict())
    if fx.nunique() > 1 and (fx.max() - fx.min()) > 1e-6:
        add_insight("4. Integrity", "LC and DC amounts differ by a non-constant rate — decide which currency to forecast in.", "warning")

# Document-level structure: one document can have several lines?
doc_lines = df.groupby(CFG["DOC_NO"]).size()
print(f"Documents: {doc_lines.shape[0]:,} | multi-line documents: {(doc_lines > 1).sum():,} (max lines {doc_lines.max()})")
kpi("n_documents", int(doc_lines.shape[0]))

## 5. Feature engineering
Features built here are reused in later sections **and** are candidates for the model.

| Feature | Definition | Why it matters |
|---|---|---|
| `terms_days` | Due − Baseline | Contractual payment terms (30/45/60…) |
| `baseline_gap` | Baseline − Posting | Posting lag vs. terms start |
| `days_to_due` | Due − Posting | Planned collection horizon |
| `days_to_clear` | Clearing − Posting | Actual collection horizon |
| `payment_delay` | Clearing − Due | **Core target of behaviour**; <0 early, >0 late |
| `is_open` | no clearing date | Right-censored observations |
| calendar features | dow, dom, month, month-end flags… | Payment timing is calendar-driven |
| `age_overdue` | DATA_END − Due (open items) | Ageing bucket of unpaid items |

In [ ]:
df["terms_days"]    = (df[D] - df[B]).dt.days
df["baseline_gap"]  = (df[B] - df[P]).dt.days
df["days_to_due"]   = (df[D] - df[P]).dt.days
df["days_to_clear"] = (df[C] - df[P]).dt.days
df["payment_delay"] = (df[C] - df[D]).dt.days
df["is_open"]       = df[C].isna()
df["is_late"]       = (df["payment_delay"] > 0).astype("float").where(df["payment_delay"].notna())
df["abs_amount"]    = df[AMT].abs()
df["log_abs_amount"]= np.log1p(df["abs_amount"])
df["age_overdue"]   = np.where(df["is_open"], (DATA_END - df[D]).dt.days, np.nan)

def add_cal(d, col, pre):
    s = d[col]
    d[f"{pre}_dow"] = s.dt.dayofweek
    d[f"{pre}_dom"] = s.dt.day
    d[f"{pre}_week_of_month"] = (s.dt.day - 1) // 7 + 1
    d[f"{pre}_month"] = s.dt.month
    d[f"{pre}_quarter"] = s.dt.quarter
    d[f"{pre}_is_month_start"] = s.dt.is_month_start
    d[f"{pre}_is_month_end"] = s.dt.is_month_end
    d[f"{pre}_days_to_month_end"] = s.dt.days_in_month - s.dt.day
    d[f"{pre}_is_weekend"] = s.dt.dayofweek >= 5
for col, pre in [(P, "post"), (D, "due"), (C, "clr")]:
    add_cal(df, col, pre)

df["amount_bucket"] = pd.qcut(df["abs_amount"].rank(method="first"), 5, labels=["Q1 small", "Q2", "Q3", "Q4", "Q5 large"])
df["ageing_bucket"] = pd.cut(df["age_overdue"], [-9999, 0, 30, 60, 90, 180, 99999],
                             labels=["not yet due", "1-30", "31-60", "61-90", "91-180", "180+"])

feature_summary = df[["terms_days", "baseline_gap", "days_to_due", "days_to_clear", "payment_delay"]].describe(percentiles=[.05, .25, .5, .75, .95]).T
add_table("feature_summary_delays", feature_summary, "Distribution of term / lag features (days)", "5. Features")
kpi("pct_open_items", float(df.is_open.mean())); kpi("pct_open_amount", float(df.loc[df.is_open, AMT].sum() / df[AMT].sum()))
add_insight("5. Features", f"{df.is_open.mean():.1%} of lines (and {df.loc[df.is_open, AMT].sum() / df[AMT].sum():.1%} of amount) are still open at data end.")
feature_summary

In [ ]:
# Payment terms mix
terms = df["terms_days"].value_counts(dropna=False).rename_axis("terms_days").to_frame("rows")
terms["share"] = terms.rows / terms.rows.sum()
terms["amount_share"] = df.groupby("terms_days")[AMT].sum().reindex(terms.index) / df[AMT].sum()
add_table("payment_terms_mix", terms.head(15), "Payment terms (Due - Baseline) mix", "5. Features")
display(terms.head(15))

## 6. Categorical profile
`Business_Area` (0131 / 0019), `Posting_Key` (01 / 05 / 07), `GL_Document_Type` (F2, AB, DR, ZM, S2, ZP), `HBKID` (house bank, 73 values).
For each: volume, amount share, average delay, % late, % open. Differences here tell us which dimensions deserve their **own model or feature**.

In [ ]:
def cat_profile(d, col):
    g = d.groupby(col, dropna=False).agg(
        rows=(AMT, "size"), amount=(AMT, "sum"), avg_amount=(AMT, "mean"), median_amount=(AMT, "median"),
        median_terms=("terms_days", "median"), median_delay=("payment_delay", "median"),
        p90_delay=("payment_delay", lambda x: x.quantile(.9)), pct_late=("is_late", "mean"), pct_open=("is_open", "mean"))
    g["row_share"] = g.rows / g.rows.sum()
    g["amount_share"] = g.amount / g.amount.sum()
    return g.sort_values("amount", ascending=False)

cat_tables = {}
for col in CAT:
    t = cat_profile(df, col); cat_tables[col] = t
    add_table(f"cat_profile_{col}", t.head(20), f"Profile by {col} (top 20 by amount)", "6. Categorical")
    print(f"\n=== {col} ({t.shape[0]} values) ===")
    display(t.head(15))

fig, axes = plt.subplots(1, len(CAT), figsize=(5 * len(CAT), 4))
for ax, col in zip(np.atleast_1d(axes), CAT):
    t = cat_tables[col].head(12)
    ax.bar(t.index.astype(str), t.amount_share); ax.set_title(f"Amount share by {col}"); ax.tick_params(axis="x", rotation=60)
save_fig("cat_amount_share", "Share of total amount by categorical dimension", "6. Categorical")

In [ ]:
# Cross-tab: Document type x Posting key (what combos exist, and how they behave)
xt = df.pivot_table(index="GL_Document_Type", columns="Posting_Key", values=AMT, aggfunc=["size", "sum"], fill_value=0)
add_table("doctype_x_postingkey", xt, "GL_Document_Type x Posting_Key (rows and amount)", "6. Categorical")
display(xt)

# Sign convention per doc type / posting key — critical to decide what is "cash in" vs "credit note"
sign_tab = df.assign(sign=np.sign(df[AMT]).map({-1: "negative", 0: "zero", 1: "positive"})).pivot_table(
    index=["GL_Document_Type", "Posting_Key"], columns="sign", values=AMT, aggfunc="size", fill_value=0)
add_table("sign_by_doctype_postingkey", sign_tab, "Sign of Amount by doc type / posting key", "6. Categorical")
display(sign_tab)
add_insight("6. Categorical", "Use the sign table above to decide which document types are invoices (+) vs credit memos / reversals (-) vs payments; "
            "decide whether credit memos reduce expected inflow or are modelled separately.", "action")

## 7. Amount analysis
Cash amounts are almost always **heavy-tailed**: a few huge items dominate totals. This decides the loss function (MAE/WAPE vs RMSE), whether to model in log space, and whether big items need manual treatment.

In [ ]:
amt_desc = df[AMT].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).to_frame("Amount")
amt_desc.loc["skew"] = df[AMT].skew(); amt_desc.loc["kurtosis"] = df[AMT].kurt()
add_table("amount_describe", amt_desc, "Amount_in_LC distribution", "7. Amounts")
display(amt_desc)

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
sns.histplot(df[AMT].clip(df[AMT].quantile(.01), df[AMT].quantile(.99)), bins=60, ax=ax[0]); ax[0].set_title("Amount (clipped 1-99%)")
pos = df.loc[df[AMT] > 0, AMT]
sns.histplot(np.log10(pos), bins=60, ax=ax[1]); ax[1].set_title("log10(Amount) — positive items")
sns.boxplot(data=df, x="GL_Document_Type", y="log_abs_amount", ax=ax[2]); ax[2].set_title("log(|Amount|) by doc type")
save_fig("amount_distribution", "Amount distribution (raw, log, by doc type)", "7. Amounts")

# Outliers (IQR on log|amount|) + concentration of top items
q1, q3 = df.log_abs_amount.quantile([.25, .75]); iqr = q3 - q1
df["is_amount_outlier"] = (df.log_abs_amount > q3 + 3 * iqr) | (df.log_abs_amount < q1 - 3 * iqr)
srt = df.abs_amount.sort_values(ascending=False)
top1 = srt.head(max(1, int(len(df) * .01))).sum() / srt.sum()
top10 = srt.head(10).sum() / srt.sum()
kpi("top1pct_items_share_of_abs_amount", float(top1)); kpi("top10_items_share", float(top10))
add_insight("7. Amounts", f"Top 1% of lines carry {top1:.1%} of absolute amount; top 10 lines carry {top10:.1%}. "
            f"{int(df.is_amount_outlier.sum())} extreme outliers (3xIQR on log scale).", "info")
top_items = df.nlargest(20, "abs_amount")[[CFG["DOC_NO"], CUST, "GL_Document_Type", "Posting_Key", P, D, C, AMT, "payment_delay"]]
add_table("top20_items", top_items, "20 largest line items by absolute amount", "7. Amounts", index=False)
top_items

## 8. Time-series construction
We build the **three series** (daily / weekly / monthly):

* **Billing** (by `Posting_Date`) — activity driver
* **Expected cash** (by `Due_Date`) — contractual schedule
* **Actual cash** (by `Clearing_Posting_Date`, cleared items only) — **forecast target**

Two traps handled below:
1. **Missing dates** — transaction data has no row on days with no activity; we reindex to a full calendar and fill zeros.
2. **Right-censoring** — recent invoices that haven't been paid yet are *not* in the clearing series, so the most recent posting cohorts look better than they are. Also the last partial week/month must be dropped.

In [ ]:
def make_series(d, date_col, value=AMT, freq="D", agg="sum", start=None, end=None):
    s = d.dropna(subset=[date_col]).set_index(date_col)[value].resample(freq).agg(agg)
    lo = start if start is not None else s.index.min()
    hi = end if end is not None else s.index.max()
    full = pd.date_range(lo, hi, freq=freq)
    return s.reindex(full, fill_value=0.0)

def trim_partial(s, freq):
    """Drop last bucket if it extends beyond DATA_END (incomplete period)."""
    if len(s) and freq == "W-SUN" and s.index[-1] > DATA_END: s = s.iloc[:-1]
    if len(s) and freq == "MS" and (s.index[-1] + pd.offsets.MonthEnd(0)) > DATA_END: s = s.iloc[:-1]
    return s

cleared = df[df[C].notna() & (df[C] <= DATA_END)]
SER = {}
for freq, tag in [("D", "daily"), ("W-SUN", "weekly"), ("MS", "monthly")]:
    SER[("billing", tag)]  = trim_partial(make_series(df, P, freq=freq, end=DATA_END), freq)
    SER[("clearing", tag)] = trim_partial(make_series(cleared, C, freq=freq, end=DATA_END), freq)
    SER[("due", tag)]      = make_series(df, D, freq=freq)                   # extends into the future (open schedule)
clr_d, clr_w, clr_m = SER[("clearing", "daily")], SER[("clearing", "weekly")], SER[("clearing", "monthly")]
bil_w, due_w = SER[("billing", "weekly")], SER[("due", "weekly")]

cov = pd.DataFrame({k[0] + "_" + k[1]: {"start": v.index.min().date(), "end": v.index.max().date(), "n_periods": len(v),
                                         "zero_periods_pct": round((v == 0).mean() * 100, 1), "total": v.sum()}
                    for k, v in SER.items()}).T
add_table("series_coverage", cov, "Coverage of the constructed series", "8. Time series")
cov

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(14, 10), sharex=False)
for a, (kind, title) in zip(ax, [("billing", "Billing (by Posting_Date)"), ("due", "Expected cash (by Due_Date)"), ("clearing", "Actual cash (by Clearing date)")]):
    SER[(kind, "weekly")].plot(ax=a, lw=1.5); SER[(kind, "monthly")].plot(ax=a, lw=2, marker="o", alpha=.6, secondary_y=False)
    a.set_title(title + " — weekly (thin) & monthly (thick)"); a.set_ylabel(AMT)
save_fig("three_series", "Billing vs expected vs actual cash", "8. Time series")

# overlay
pd.concat({"billing": bil_w, "due": due_w, "clearing": clr_w}, axis=1).loc[clr_w.index.min():clr_w.index.max()].plot(figsize=(14, 4))
plt.title("Weekly overlay: billing, expected (due), actual (clearing)")
save_fig("overlay_weekly", "Overlay of the three weekly series", "8. Time series")

In [ ]:
# Right-censoring check: for each posting-month cohort, how much is already cleared?
coh = df.assign(post_ym=df[P].dt.to_period("M").astype(str)).groupby("post_ym").agg(
    rows=(AMT, "size"), amount=(AMT, "sum"), pct_rows_cleared=("is_open", lambda x: 1 - x.mean()),
    mean_delay_of_cleared=("payment_delay", "mean"))
coh["pct_amount_open"] = df.assign(post_ym=df[P].dt.to_period("M").astype(str)).assign(o=lambda x: x[AMT] * x.is_open).groupby("post_ym").o.sum() / coh.amount
add_table("posting_cohort_censoring", coh, "Share cleared per posting-month cohort (right-censoring check)", "8. Time series")
display(coh)
coh["pct_rows_cleared"].plot(kind="bar", figsize=(12, 3), color="steelblue"); plt.title("% of lines cleared by posting month cohort"); plt.ylim(0, 1)
save_fig("cohort_cleared", "Cleared share per posting cohort — recent cohorts are censored", "8. Time series")
add_insight("8. Time series", "Recent posting cohorts are only partially cleared. Never compute payment-delay statistics on them without "
            "restricting to mature items (see MATURE_DAYS) — otherwise delay is biased low.", "warning")

## 9. Trend & seasonality of actual cash (the forecast target)
With only ~1.5 years there are fewer than 2 full yearly cycles, so **annual seasonality cannot be estimated reliably** — we focus on *intra-week* (weekday), *intra-month* (day-of-month / month-end), and visible month-to-month patterns. These are the cycles a model can actually learn.

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(14, 7))
clr_d.plot(ax=ax[0], lw=.8); clr_d.rolling(28).mean().plot(ax=ax[0], lw=2, color="crimson", label="28d mean")
ax[0].set_title("Daily actual cash with 28-day rolling mean"); ax[0].legend()
clr_w.plot(ax=ax[1], marker=".", lw=1); clr_w.rolling(4).mean().plot(ax=ax[1], lw=2, color="crimson", label="4w mean"); clr_w.rolling(4).std().plot(ax=ax[1], lw=1.5, color="green", label="4w std")
ax[1].set_title("Weekly actual cash: level (mean) & volatility (std)"); ax[1].legend()
save_fig("clearing_trend", "Daily and weekly actual cash with rolling statistics", "9. Trend/Seasonality")

In [ ]:
d = pd.DataFrame({"y": clr_d})
d["dow"] = d.index.dayofweek; d["dom"] = d.index.day; d["wom"] = (d.index.day - 1) // 7 + 1; d["month"] = d.index.month
d["is_month_end_window"] = d.index.day >= (d.index.days_in_month - 2)
d["is_business_day"] = d.index.dayofweek < 5

fig, ax = plt.subplots(2, 2, figsize=(15, 8))
dow = d.groupby("dow").y.agg(["mean", "median", "sum"]); dow.index = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
dow["mean"].plot.bar(ax=ax[0, 0], color="steelblue"); ax[0, 0].set_title("Mean daily cash by weekday")
dom = d.groupby("dom").y.mean(); dom.plot.bar(ax=ax[0, 1], color="darkorange"); ax[0, 1].set_title("Mean daily cash by day-of-month")
wom = d.groupby("wom").y.mean(); wom.plot.bar(ax=ax[1, 0], color="seagreen"); ax[1, 0].set_title("Mean daily cash by week-of-month")
mon = clr_m.to_frame("y"); mon.index = mon.index.strftime("%Y-%m"); mon["y"].plot.bar(ax=ax[1, 1], color="slateblue"); ax[1, 1].set_title("Monthly cash (full months)")
save_fig("seasonality_profiles", "Weekday / day-of-month / week-of-month / monthly profiles", "9. Trend/Seasonality")

add_table("weekday_profile", dow, "Cash by weekday", "9. Trend/Seasonality")
add_table("dayofmonth_profile", dom.to_frame("mean_daily_cash"), "Cash by day of month", "9. Trend/Seasonality")
wk_share = float(d.loc[~d.is_business_day, "y"].abs().sum() / d.y.abs().sum())
kpi("weekend_cash_share", wk_share)
peak_dow = dow["mean"].idxmax(); peak_dom = int(dom.idxmax())
me_ratio = float(d.loc[d.is_month_end_window, "y"].mean() / d.loc[~d.is_month_end_window, "y"].mean())
kpi("month_end_window_ratio", me_ratio)
add_insight("9. Trend/Seasonality", f"Weekend cash share {wk_share:.1%}; strongest weekday = {peak_dow}; strongest day-of-month = {peak_dom}; "
            f"last-3-days-of-month average is {me_ratio:.2f}x the rest of the month.")
if wk_share < .02:
    add_insight("9. Trend/Seasonality", "Almost no weekend cash — model on business days or add a weekend flag; zeros inflate daily error metrics.", "action")
display(dow)

In [ ]:
# Heatmap: week-of-month x weekday (when in the month does cash land?) and month x weekday
hm = d.pivot_table(index="wom", columns="dow", values="y", aggfunc="mean")
hm.columns = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"][:hm.shape[1]]
plt.figure(figsize=(8, 4)); sns.heatmap(hm, annot=True, fmt=",.0f", cmap="YlGnBu"); plt.title("Mean daily cash: week-of-month x weekday")
save_fig("heatmap_wom_dow", "Heatmap of average daily cash by week-of-month and weekday", "9. Trend/Seasonality")

# STL decomposition on business-day-agnostic daily series (period = 7)
if len(clr_d) >= 28:
    stl = STL(clr_d.astype(float), period=7, robust=True).fit()
    fig = stl.plot(); fig.set_size_inches(14, 8); fig.suptitle("STL decomposition (daily, period=7)", y=1.01)
    save_fig("stl_daily", "STL decomposition of daily actual cash", "9. Trend/Seasonality")
    var = lambda x: np.var(x)
    Ft = max(0, 1 - var(stl.resid) / var(stl.trend + stl.resid)); Fs = max(0, 1 - var(stl.resid) / var(stl.seasonal + stl.resid))
    kpi("trend_strength_daily", float(Ft)); kpi("weekly_seasonal_strength_daily", float(Fs))
    add_insight("9. Trend/Seasonality", f"STL strengths (0-1): trend {Ft:.2f}, weekly seasonality {Fs:.2f}.")

## 10. Statistical diagnostics
* **ADF** (H0 = unit root, i.e. non-stationary) and **KPSS** (H0 = stationary) — use both; if they disagree, the series is probably trend-stationary or needs differencing.
* **ACF / PACF** — reveal autoregressive memory and seasonal lags (7 for daily, 4–5 for monthly cycles on weekly data).
* **Ljung-Box** — tests whether the series has any autocorrelation at all (if not, a simple mean is as good as any model).
* **ADI / CV²** (Syntetos–Boylan) — classifies demand as smooth / erratic / intermittent / lumpy, which decides model family.

In [ ]:
def stationarity(s, name):
    s = s.astype(float).dropna()
    out = {"series": name, "n": len(s)}
    try:
        a = adfuller(s, autolag="AIC"); out.update(adf_stat=a[0], adf_p=a[1])
    except Exception as e: out.update(adf_stat=np.nan, adf_p=np.nan)
    try:
        k = kpss(s, regression="c", nlags="auto"); out.update(kpss_stat=k[0], kpss_p=k[1])
    except Exception as e: out.update(kpss_stat=np.nan, kpss_p=np.nan)
    out["verdict"] = ("stationary" if out["adf_p"] < .05 and out["kpss_p"] >= .05 else
                      "non-stationary" if out["adf_p"] >= .05 and out["kpss_p"] < .05 else "ambiguous (try differencing)")
    return out

tests = pd.DataFrame([stationarity(clr_w, "weekly clearing"), stationarity(np.log1p(clr_w.clip(lower=0)), "weekly clearing (log1p)"),
                      stationarity(clr_w.diff().dropna(), "weekly clearing (diff 1)"), stationarity(clr_d, "daily clearing")]).set_index("series")
add_table("stationarity_tests", tests, "ADF / KPSS tests", "10. Diagnostics")
display(tests)

lb = acorr_ljungbox(clr_w.astype(float), lags=[4, 8, 12], return_df=True)
add_table("ljung_box_weekly", lb, "Ljung-Box test, weekly clearing", "10. Diagnostics")
display(lb)
kpi("ljung_box_p_lag8", float(lb.loc[8, "lb_pvalue"]))

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(15, 7))
plot_acf(clr_d.astype(float), lags=min(40, len(clr_d) // 2 - 1), ax=ax[0, 0]); ax[0, 0].set_title("ACF — daily")
plot_pacf(clr_d.astype(float), lags=min(40, len(clr_d) // 2 - 1), ax=ax[0, 1], method="ywm"); ax[0, 1].set_title("PACF — daily")
nl = min(26, len(clr_w) // 2 - 1)
plot_acf(clr_w.astype(float), lags=nl, ax=ax[1, 0]); ax[1, 0].set_title("ACF — weekly")
plot_pacf(clr_w.astype(float), lags=nl, ax=ax[1, 1], method="ywm"); ax[1, 1].set_title("PACF — weekly")
save_fig("acf_pacf", "ACF / PACF of daily and weekly actual cash", "10. Diagnostics")

def adi_cv2(s):
    nz = s[s != 0]
    if len(nz) < 2: return np.nan, np.nan
    return len(s) / len(nz), (nz.std(ddof=0) / abs(nz.mean())) ** 2
def demand_class(adi, cv2):
    if np.isnan(adi): return "insufficient"
    if adi < 1.32: return "smooth" if cv2 < .49 else "erratic"
    return "intermittent" if cv2 < .49 else "lumpy"

rows = []
for name, s in [("daily clearing", clr_d), ("business-day clearing", clr_d[clr_d.index.dayofweek < 5]), ("weekly clearing", clr_w), ("monthly clearing", clr_m)]:
    a, c2 = adi_cv2(s); rows.append({"series": name, "ADI": a, "CV2": c2, "class": demand_class(a, c2), "zero_pct": (s == 0).mean() * 100})
inter = pd.DataFrame(rows).set_index("series")
add_table("intermittency_total", inter, "ADI/CV² demand classification of total cash series", "10. Diagnostics")
display(inter)

## 11. Payment behaviour (the most important section for cash forecasting)
**Collection curve** = % of invoice amount collected *k days relative to due date*. If we know what falls due and this curve, we can already forecast cash.
To avoid censoring bias, the curve is built on **mature** items only (due ≥ `MATURE_DAYS` before data end); items still open count as "not collected yet".

In [ ]:
mat = df[(df[D] <= DATA_END - pd.Timedelta(days=CFG["MATURE_DAYS"])) & (df[AMT] > 0)].copy()
print(f"Mature positive items: {len(mat):,} of {len(df):,}")

def collection_curve(m, lo=-30, hi=CFG["MATURE_DAYS"]):
    tot = m[AMT].sum()
    ks = np.arange(lo, hi + 1)
    return pd.Series([m.loc[m.payment_delay <= k, AMT].sum() / tot if tot else np.nan for k in ks], index=ks, name="cum_share")

curve_all = collection_curve(mat)
curve_ba = {b: collection_curve(g) for b, g in mat.groupby("Business_Area")}
fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))
curve_all.plot(ax=ax[0], lw=2.5, color="black", label="All")
for b, c in curve_ba.items(): c.plot(ax=ax[0], label=f"BA {b}")
ax[0].axvline(0, color="red", ls="--"); ax[0].set_title("Amount-weighted collection curve (days vs due date)"); ax[0].set_xlabel("days relative to due"); ax[0].legend()
sns.histplot(mat["payment_delay"].dropna().clip(-30, 120), bins=75, ax=ax[1]); ax[1].axvline(0, color="red", ls="--"); ax[1].set_title("Payment delay distribution (cleared mature items, clipped)")
save_fig("collection_curve", "Collection curve and delay histogram", "11. Payment behaviour")

milestones = pd.Series({f"collected by due+{k}d": curve_all.get(k, np.nan) for k in [-7, 0, 7, 14, 30, 60, 90]})
add_table("collection_milestones", milestones.to_frame("share_of_amount"), "Share of mature amount collected by milestone", "11. Payment behaviour")
for k, v in milestones.items(): kpi(k.replace(" ", "_").replace("+", "_plus_"), float(v))
med = mat["payment_delay"].median(); p90 = mat["payment_delay"].quantile(.9)
kpi("median_delay_days", float(med)); kpi("p90_delay_days", float(p90))
add_insight("11. Payment behaviour", f"Median payment delay = {med:.0f} days, P90 = {p90:.0f} days. On-time-or-early share of amount: {curve_all.get(0, np.nan):.1%}; "
            f"collected within 30 days after due: {curve_all.get(30, np.nan):.1%}.")
milestones

In [ ]:
# Delay drivers: by customer / category (mature, cleared items)
mc = mat[mat.payment_delay.notna()]
def delay_table(g):
    return pd.Series({"rows": len(g), "amount": g[AMT].sum(), "mean_delay": g.payment_delay.mean(), "median_delay": g.payment_delay.median(),
                      "std_delay": g.payment_delay.std(), "p90_delay": g.payment_delay.quantile(.9),
                      "pct_late": (g.payment_delay > 0).mean(),
                      "wtd_delay": np.average(g.payment_delay, weights=g[AMT].clip(lower=1e-9))})
cust_beh = mc.groupby(CUST).apply(delay_table).sort_values("amount", ascending=False)
cust_beh["behaviour"] = np.select(
    [cust_beh.rows < 5, cust_beh.median_delay <= 0, cust_beh.median_delay <= 15, cust_beh.median_delay > 15],
    ["too few items", "prompt/early", "moderately late", "chronically late"], default="n/a")
cust_beh["volatile"] = cust_beh.std_delay > cust_beh.std_delay.median() * 1.5
add_table("customer_payment_behaviour", cust_beh.head(40), "Payment behaviour per customer (mature cleared items)", "11. Payment behaviour")
display(cust_beh.head(15))
print(cust_beh.behaviour.value_counts())

for col in ["Business_Area", "GL_Document_Type", "Posting_Key", "terms_days"]:
    t = mc.groupby(col).apply(delay_table).sort_values("amount", ascending=False)
    add_table(f"delay_by_{col}", t.head(15), f"Delay statistics by {col}", "11. Payment behaviour")
    print(f"\n--- Delay by {col} ---"); display(t.head(10))

# is delay really customer-driven? variance share (eta^2)
grp_mean = mc.groupby(CUST).payment_delay.transform("mean")
eta2 = 1 - ((mc.payment_delay - grp_mean) ** 2).sum() / ((mc.payment_delay - mc.payment_delay.mean()) ** 2).sum()
kpi("delay_variance_explained_by_customer", float(eta2))
add_insight("11. Payment behaviour", f"Customer identity explains {eta2:.0%} of payment-delay variance — "
            + ("customer-level features (historic mean/std delay) are essential." if eta2 > .25 else "customer effect is modest; look at other drivers."), "action" if eta2 > .25 else "info")

In [ ]:
# Delay stability over time (is behaviour drifting?) — by DUE month (mature only) to avoid censoring bias
trend = mc.assign(due_ym=mc[D].dt.to_period("M").astype(str)).groupby("due_ym").payment_delay.agg(["count", "mean", "median", lambda x: x.quantile(.9)])
trend.columns = ["n", "mean", "median", "p90"]
add_table("delay_trend_by_due_month", trend, "Monthly payment delay trend (by due month)", "11. Payment behaviour")
trend[["mean", "median", "p90"]].plot(figsize=(12, 3.5), marker="o"); plt.axhline(0, color="red", ls="--"); plt.title("Delay by due month — drift check")
save_fig("delay_drift", "Delay drift over time", "11. Payment behaviour")

# Which weekday / day-of-month do payments land on, relative to due-date weekday?
lag_dow = mc.pivot_table(index="due_dow", columns="clr_dow", values=AMT, aggfunc="size", fill_value=0)
plt.figure(figsize=(7, 4)); sns.heatmap(lag_dow.div(lag_dow.sum(1), axis=0), annot=True, fmt=".2f", cmap="Blues"); plt.title("P(clearing weekday | due weekday)")
save_fig("due_vs_clear_weekday", "Clearing weekday given due weekday", "11. Payment behaviour")

# Amount vs delay: do big invoices pay later?
rho, pval = stats.spearmanr(mc["log_abs_amount"], mc["payment_delay"])
kpi("spearman_amount_vs_delay", float(rho))
add_insight("11. Payment behaviour", f"Spearman(amount, delay) = {rho:.2f} (p={pval:.3f}).")

## 12. Customer analysis
Concentration risk (Pareto / HHI), customer activity over time, and per-customer demand class. Highly concentrated portfolios are better handled **bottom-up for top customers + top-down for the tail**.

In [ ]:
cs = df.groupby(CUST).agg(rows=(AMT, "size"), amount=(AMT, "sum"), first_post=(P, "min"), last_post=(P, "max")).sort_values("amount", ascending=False)
cs["share"] = cs.amount / cs.amount.sum(); cs["cum_share"] = cs.share.cumsum()
hhi = float((cs.share ** 2).sum())
n80 = int((cs.cum_share < .8).sum() + 1)
kpi("n_customers", int(len(cs))); kpi("customers_for_80pct_amount", n80); kpi("customer_hhi", hhi)
add_insight("12. Customers", f"{len(cs)} customers; {n80} customers make 80% of amount. Top customer share {cs.share.iloc[0]:.1%}; HHI={hhi:.3f}.")
add_table("customer_concentration_top", cs.head(CFG["TOP_N"] * 2), "Top customers by amount", "12. Customers")

fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))
ax[0].plot(np.arange(1, len(cs) + 1), cs.cum_share.values, marker="o"); ax[0].axhline(.8, color="red", ls="--"); ax[0].set_title("Pareto: cumulative amount share vs # customers")
cs.share.head(15).plot.bar(ax=ax[1]); ax[1].set_title("Top-15 customer share")
save_fig("customer_pareto", "Customer concentration", "12. Customers")
display(cs.head(15))

In [ ]:
# Customer x month activity heatmap (clearing), active-customer count
cm = cleared.assign(ym=cleared[C].dt.to_period("M").astype(str)).pivot_table(index=CUST, columns="ym", values=AMT, aggfunc="sum", fill_value=0)
top_c = cs.head(20).index
plt.figure(figsize=(14, 6)); sns.heatmap(np.log1p(cm.loc[cm.index.intersection(top_c)].clip(lower=0)), cmap="YlOrRd", cbar_kws={"label": "log1p(cash)"})
plt.title("Top-20 customers: monthly cash received (log scale)")
save_fig("customer_month_heatmap", "Customer x month cash heatmap", "12. Customers")

active = (cm > 0).sum(); active.plot(kind="bar", figsize=(12, 3), color="teal"); plt.title("# customers paying per month")
save_fig("active_customers", "Active paying customers per month", "12. Customers")

# Per-customer weekly series classification (smooth / erratic / intermittent / lumpy)
weeks = clr_w.index
cw = cleared.groupby([CUST, pd.Grouper(key=C, freq="W-SUN")])[AMT].sum().unstack(0).reindex(weeks).fillna(0)
cls = pd.DataFrame([(c, *adi_cv2(cw[c])) for c in cw.columns], columns=[CUST, "ADI", "CV2"]).set_index(CUST)
cls["class"] = [demand_class(a, c) for a, c in zip(cls.ADI, cls.CV2)]
cls["amount"] = cs.amount.reindex(cls.index)
class_mix = cls.groupby("class").agg(n_customers=("ADI", "size"), amount=("amount", "sum")); class_mix["amount_share"] = class_mix.amount / class_mix.amount.sum()
add_table("customer_demand_class_mix", class_mix, "Customers by weekly demand class (ADI/CV²)", "12. Customers")
display(class_mix)
plt.figure(figsize=(7, 5)); sns.scatterplot(data=cls.reset_index(), x="ADI", y="CV2", hue="class", size="amount", sizes=(30, 300), alpha=.7)
plt.axvline(1.32, color="grey", ls="--"); plt.axhline(.49, color="grey", ls="--"); plt.title("Syntetos-Boylan map (weekly, per customer)")
save_fig("sb_map", "ADI/CV² classification map per customer", "12. Customers")
share_il = float(class_mix.reindex(["intermittent", "lumpy"]).amount_share.fillna(0).sum())
add_insight("12. Customers", f"{share_il:.0%} of amount sits in customers with intermittent/lumpy weekly cash. "
            + ("Forecast at an aggregate level (or via due-date schedule x delay distribution) instead of one model per customer." if share_il > .5 else "Per-customer models are feasible for the smooth/erratic customers."), "action")

## 13. Segment-level series
Compare Business_Area / Document type / House bank series: do they move together (use top-down + reconciliation) or independently (bottom-up)?

In [ ]:
def seg_series(col, top=6):
    keys = cleared.groupby(col)[AMT].sum().abs().sort_values(ascending=False).head(top).index
    out = {}
    for k in keys:
        out[str(k)] = trim_partial(make_series(cleared[cleared[col] == k], C, freq="W-SUN", end=DATA_END), "W-SUN").reindex(clr_w.index, fill_value=0)
    return pd.DataFrame(out)

seg_tables = {}
for col in ["Business_Area", "GL_Document_Type", "HBKID"]:
    sw = seg_series(col); seg_tables[col] = sw
    fig, ax = plt.subplots(1, 2, figsize=(16, 4))
    sw.plot(ax=ax[0], lw=1); ax[0].set_title(f"Weekly actual cash by {col}")
    sns.heatmap(sw.corr(), annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, ax=ax[1]); ax[1].set_title(f"Correlation between {col} segments")
    save_fig(f"segments_{col}", f"Segment weekly series and correlation by {col}", "13. Segments")
    zero = (sw == 0).mean().round(2).to_frame("zero_week_share"); add_table(f"segment_zero_share_{col}", zero, "", "13. Segments"); display(zero.T)
add_insight("13. Segments", "Segments with high zero-week share are intermittent — aggregate them (e.g. 'other') before modelling; strongly correlated segments suggest a single top-down model is adequate.", "action")

## 14. Lead–lag structure & open-item (AR) balance
* **Cross-correlation** between weekly *due* amounts and weekly *clearing* amounts shows the typical lag (in weeks) between expected and actual cash — the single most predictive feature set for the model.
* **AR balance** = cumulative billed − cumulative cleared. A rising balance with flat billing means customers are paying slower (liquidity pressure).

In [ ]:
due_al = due_w.reindex(clr_w.index, fill_value=0)
bil_al = bil_w.reindex(clr_w.index, fill_value=0)
lags = range(0, 13)
cc_due = pd.Series({k: clr_w.corr(due_al.shift(k)) for k in lags}, name="corr(clearing_t, due_{t-k})")
cc_bil = pd.Series({k: clr_w.corr(bil_al.shift(k)) for k in lags}, name="corr(clearing_t, billing_{t-k})")
cc = pd.concat([cc_due, cc_bil], axis=1); cc.index.name = "lag_weeks"
add_table("lead_lag_correlation", cc, "Cross-correlation of weekly actual cash with lagged due and billing amounts", "14. Lead-lag")
cc.plot.bar(figsize=(12, 3.5)); plt.title("Cross-correlation: which lag of due/billing best explains cash?")
save_fig("lead_lag", "Lead-lag correlation", "14. Lead-lag")
best_due_lag = int(cc_due.idxmax()); best_bil_lag = int(cc_bil.idxmax())
kpi("best_due_lag_weeks", best_due_lag); kpi("best_billing_lag_weeks", best_bil_lag)
add_insight("14. Lead-lag", f"Actual cash correlates most with Due amounts lagged {best_due_lag} week(s) (r={cc_due.max():.2f}) and with billing lagged {best_bil_lag} week(s) (r={cc_bil.max():.2f}).", "action")

# AR balance
idx = pd.date_range(df[P].min(), DATA_END)
posted = df.groupby(P)[AMT].sum().reindex(idx, fill_value=0).cumsum()
paid = cleared.groupby(C)[AMT].sum().reindex(idx, fill_value=0).cumsum()
ar = (posted - paid)
fig, ax = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
ar.plot(ax=ax[0], color="firebrick"); ax[0].set_title("Open AR balance (cumulative billed − cumulative cleared)")
# DSO-like: AR / trailing 90d billing * 90
bill_90 = df.groupby(P)[AMT].sum().reindex(idx, fill_value=0).rolling(90).sum()
(ar / bill_90 * 90).plot(ax=ax[1], color="navy"); ax[1].set_title("DSO proxy (days) = AR / trailing-90d billing x 90")
save_fig("ar_balance", "AR balance and DSO proxy", "14. Lead-lag")
kpi("ar_balance_end", float(ar.iloc[-1])); kpi("dso_proxy_end", float((ar / bill_90 * 90).iloc[-1]))

# Ageing of currently open items
ageing = df[df.is_open].groupby("ageing_bucket", observed=False)[AMT].agg(["size", "sum"]); ageing["share"] = ageing["sum"] / ageing["sum"].sum()
add_table("open_items_ageing", ageing, "Ageing of open items at data end", "14. Lead-lag")
display(ageing)

## 15. Anomalies & suspicious records
* **Spike days** — robust z-score using rolling median and MAD (resistant to the spikes themselves).
* **Dominant items** — is a spike one giant payment? (then it is an event, not a pattern.)
* **Potential duplicates** — same customer, amount and due date.

In [ ]:
def robust_flags(s, win=28, thr=4.0):
    med = s.rolling(win, center=True, min_periods=7).median()
    mad = (s - med).abs().rolling(win, center=True, min_periods=7).median()
    z = 0.6745 * (s - med) / mad.replace(0, np.nan)
    return z

z = robust_flags(clr_d[clr_d.index.dayofweek < 5])
spikes = z[z.abs() > 4].dropna()
sp = pd.DataFrame({"cash": clr_d.reindex(spikes.index), "robust_z": spikes})
# largest single item on that day and its share
top_item = cleared.groupby(C).apply(lambda g: pd.Series({"largest_item": g[AMT].max(), "n_items": len(g)}))
sp = sp.join(top_item); sp["largest_item_share"] = sp.largest_item / sp.cash
sp = sp.sort_values("robust_z", key=abs, ascending=False)
add_table("spike_days", sp.head(25), "Robust-z spike days in actual cash (business days)", "15. Anomalies")
fig, ax = plt.subplots(figsize=(14, 4)); clr_d.plot(ax=ax, lw=.7); ax.scatter(sp.index, sp.cash, color="red", zorder=5, label="spike"); ax.legend(); ax.set_title("Daily cash with flagged spikes")
save_fig("spikes", "Flagged spike days", "15. Anomalies")
kpi("n_spike_days", int(len(sp)))
add_insight("15. Anomalies", f"{len(sp)} spike days flagged. Check 'largest_item_share': values near 1 mean a single payment drives the spike (event, not seasonality).",
            "action" if len(sp) else "info")
display(sp.head(10))

dup_key = [CUST, AMT, D]
dups = df[df.duplicated(dup_key, keep=False)].sort_values(dup_key)
kpi("potential_duplicate_lines", int(len(dups)))
add_table("potential_duplicates_sample", dups.head(30)[[CFG["DOC_NO"], CUST, AMT, P, D, C]], "Lines sharing customer + amount + due date", "15. Anomalies", index=False)
print(f"Lines sharing customer+amount+due date: {len(dups):,} (could be legit split lines or double postings — verify with business)")

## 16. Baseline forecasts (weekly)
Any ML/DL model must beat these. We hold out the last `HOLDOUT_WEEKS` full weeks and score with **WAPE** (= Σ|error| / Σ|actual|, robust for lumpy data), MAE and bias.
The *due-schedule baseline* shifts the weekly Due series by the best lag from §14 and rescales it — it encodes the "what falls due × typical lateness" logic.

In [ ]:
H = CFG["HOLDOUT_WEEKS"]
y = clr_w.astype(float); train, test = y.iloc[:-H], y.iloc[-H:]
preds = {}
preds["naive (last week)"] = pd.Series(train.iloc[-1], index=test.index)
preds["mean last 4w"] = pd.Series(train.iloc[-4:].mean(), index=test.index)
preds["mean last 13w"] = pd.Series(train.iloc[-13:].mean(), index=test.index)
preds["seasonal naive (4w ago)"] = pd.Series([y.shift(4).loc[t] if t in y.index else np.nan for t in test.index], index=test.index)  # uses observed lag-4; leak-free for 1-step only
# due-schedule baseline: lag chosen on TRAIN only
due_train_corr = {k: train.corr(due_al.shift(k).loc[train.index]) for k in range(0, 9)}
k_best = max(due_train_corr, key=lambda k: np.nan_to_num(due_train_corr[k], nan=-9))
sh = due_al.shift(k_best); scale = train.sum() / sh.loc[train.index].sum()
preds[f"due-schedule shifted {k_best}w x{scale:.2f}"] = (sh * scale).loc[test.index]

def metrics(a, p):
    e = p - a; return {"WAPE": e.abs().sum() / a.abs().sum(), "MAE": e.abs().mean(), "Bias%": e.sum() / a.sum() * 100}
base = pd.DataFrame({k: metrics(test, v) for k, v in preds.items()}).T.sort_values("WAPE")
add_table("baseline_metrics_weekly", base, f"Weekly baselines, last {H} weeks hold-out", "16. Baselines")
display(base)
pd.concat({"actual": test, **{k: v for k, v in list(preds.items())}}, axis=1).plot(figsize=(13, 4), marker="o"); plt.title(f"Hold-out: actual vs baselines (last {H} weeks)")
save_fig("baselines_holdout", "Baseline forecasts on hold-out", "16. Baselines")
kpi("best_baseline", base.index[0]); kpi("best_baseline_wape", float(base.WAPE.iloc[0]))
add_insight("16. Baselines", f"Best naive benchmark: '{base.index[0]}' with WAPE {base.WAPE.iloc[0]:.1%}. Target for the ML model: clearly below this on rolling-origin validation.", "action")

## 17. Modelling readiness & recommendations
Summary of what the EDA implies for the forecasting design. Everything is built from variables computed above, so it updates automatically when data changes.

In [ ]:
rec = []
n_weeks = len(clr_w)
rec.append(("History length", f"{n_weeks} full weeks ({len(clr_m)} months)",
            "Too short for yearly seasonality; use calendar/month-end features, avoid models needing >2 seasonal cycles." if n_weeks < 104 else "Enough for yearly seasonality."))
rec.append(("Target definition", "Cash by Clearing_Posting_Date (cleared items)", "Forecast at weekly (and optionally daily business-day) level in local currency."))
rec.append(("Demand class (weekly total)", inter.loc["weekly clearing", "class"], "Smooth/erratic → classical + ML ok; intermittent/lumpy → aggregate more or use probabilistic/quantile output."))
rec.append(("Stationarity", tests.loc["weekly clearing", "verdict"], "Difference or use trend features for ARIMA-type models; tree models are insensitive."))
rec.append(("Autocorrelation (Ljung-Box lag 8)", f"p={REPORT['kpis']['ljung_box_p_lag8']:.3f}", "p>0.05 → little serial memory; exogenous (due-schedule) features matter more than lags."))
rec.append(("Weekly pattern", f"weekend share {REPORT['kpis']['weekend_cash_share']:.1%}", "Model business days; add weekday & holiday calendar features."))
rec.append(("Month-end effect", f"{REPORT['kpis']['month_end_window_ratio']:.2f}x", "Add days-to-month-end, is_month_end, week-of-month features."))
rec.append(("Best due-date lag", f"{REPORT['kpis']['best_due_lag_weeks']} weeks", "Create lagged due-schedule features (due amount in t-k) + open-item snapshot features."))
rec.append(("Customer effect on delay", f"{REPORT['kpis']['delay_variance_explained_by_customer']:.0%} of variance", "Include rolling customer delay mean/std/pct-late (computed using only data before forecast origin!)."))
rec.append(("Concentration", f"{REPORT['kpis']['customers_for_80pct_amount']} customers = 80% amount", "Hybrid: bottom-up for top customers, top-down for tail."))
rec.append(("Amount skew", f"top 1% lines = {REPORT['kpis']['top1pct_items_share_of_abs_amount']:.0%} of amount", "Use WAPE/MAE or quantile loss; consider winsorising or handling mega-items separately."))
rec.append(("Censoring", f"{REPORT['kpis']['pct_open_items']:.1%} lines open", "Build features from mature items only; use survival/hazard-style delay modelling for open items."))
rec.append(("Benchmark to beat", f"{REPORT['kpis']['best_baseline']} (WAPE {REPORT['kpis']['best_baseline_wape']:.1%})", "Use rolling-origin cross-validation (expanding window), never random splits."))
readiness = pd.DataFrame(rec, columns=["Topic", "Finding", "Implication for modelling"]).set_index("Topic")
add_table("modelling_readiness", readiness, "EDA → modelling implications", "17. Readiness")
pd.set_option("display.max_colwidth", 140); display(readiness)

leakage = ["Clearing_Document / Clearing_Posting_Date / payment_delay / days_to_clear / is_open are only known AFTER payment → never use as features for the same item without an 'as-of forecast origin' cut-off.",
           "Customer statistics must be computed with data strictly before each forecast origin (rolling / expanding), not on the full history.",
           "Train/validation split must be chronological (rolling-origin)."]
for l in leakage: add_insight("17. Readiness", l, "warning")

## 18. Report export
Everything registered during the notebook (`REPORT["kpis"]`, `["insights"]`, `["tables"]`, `["figures"]`) is exported to:
* `eda_report.html` — self-contained, figures embedded
* `eda_tables.xlsx` — one sheet per table
* `eda_kpis.json` — headline numbers
* cleaned dataset as parquet for the modelling notebook

In [ ]:
def _df_html(frame, index=True):
    f = frame.copy()
    return f.to_html(index=index, float_format=lambda x: f"{x:,.3f}", border=0, classes="tbl", na_rep="")

def build_html_report(path=OUT / "eda_report.html"):
    sections = sorted({i["section"] for i in REPORT["insights"]} | {t["section"] for t in REPORT["tables"].values()} | {f["section"] for f in REPORT["figures"]})
    css = """body{font-family:Segoe UI,Arial;margin:30px auto;max-width:1150px;color:#222} h1{border-bottom:3px solid #1f4e79}
    h2{margin-top:40px;color:#1f4e79;border-bottom:1px solid #ccc} .tbl{border-collapse:collapse;font-size:12px;margin:8px 0}
    .tbl th{background:#1f4e79;color:#fff;padding:4px 8px} .tbl td{padding:3px 8px;border-bottom:1px solid #e5e5e5}
    .info{background:#eef5fb;padding:6px 10px;margin:4px 0;border-left:4px solid #1f77b4}.warning{background:#fff6e5;padding:6px 10px;margin:4px 0;border-left:4px solid #ff9800}
    .action{background:#eaf7ea;padding:6px 10px;margin:4px 0;border-left:4px solid #2e7d32} img{max-width:100%;margin:6px 0} .kpi td{padding:2px 12px}"""
    h = [f"<html><head><meta charset='utf-8'><title>Cash Forecasting EDA</title><style>{css}</style></head><body>",
         "<h1>Cash Liquidity Forecasting — EDA Report</h1>", f"<p>Generated: {REPORT['meta']['created']}</p>",
         "<h2>Headline numbers</h2><table class='kpi'>" + "".join(f"<tr><td><b>{k}</b></td><td>{v if not isinstance(v, float) else f'{v:,.4f}'}</td></tr>" for k, v in REPORT["kpis"].items()) + "</table>"]
    for sec in sections:
        h.append(f"<h2>{sec}</h2>")
        for i in [i for i in REPORT["insights"] if i["section"] == sec]: h.append(f"<div class='{i['level']}'>{i['text']}</div>")
        for name, t in REPORT["tables"].items():
            if t["section"] == sec: h.append(f"<h4>{name}</h4><i>{t['note']}</i>" + _df_html(t["df"], t["index"]))
        for f in REPORT["figures"]:
            if f["section"] == sec:
                b64 = base64.b64encode(open(f["path"], "rb").read()).decode()
                h.append(f"<div><img src='data:image/png;base64,{b64}'/><br><i>{f['caption']}</i></div>")
    h.append("</body></html>")
    Path(path).write_text("\n".join(h), encoding="utf-8"); return path

html_path = build_html_report()
print("HTML report ->", html_path)

# Excel with all tables
try:
    with pd.ExcelWriter(OUT / "eda_tables.xlsx") as xw:
        for name, t in REPORT["tables"].items():
            t["df"].to_excel(xw, sheet_name=name[:31], index=t["index"])
    print("Excel ->", OUT / "eda_tables.xlsx")
except Exception as e:
    print("Excel export skipped:", e)

# KPIs / insights json and cleaned data
(OUT / "eda_kpis.json").write_text(json.dumps({"kpis": REPORT["kpis"], "insights": REPORT["insights"]}, default=str, indent=2))
try:
    df.to_parquet(OUT / "clean_transactions.parquet", index=False); print("Clean data ->", OUT / "clean_transactions.parquet")
except Exception as e:
    print("Parquet skipped:", e)
print(f"\nRegistered: {len(REPORT['kpis'])} KPIs | {len(REPORT['insights'])} insights | {len(REPORT['tables'])} tables | {len(REPORT['figures'])} figures")

### Next steps after this EDA
1. Fix every `REVIEW`/`warning` item above with the business (date logic, duplicates, sign conventions).
2. Build the **feature store**: lagged due-schedule, open-item snapshots as of each forecast origin, customer delay statistics (as-of), calendar/holiday features.
3. Set up **rolling-origin validation** and reproduce the baselines in §16.
4. Then model ladder: seasonal-naive → ETS/ARIMA → gradient boosting (LightGBM) on engineered features → (optional) DL (N-BEATS/TFT) if data volume justifies.